# DINOv2 — emergent segmentation + a linear probe

**Foundation model** · supports lessons C3 (backbones) and C5/C6 (hands & interaction).

DINOv2 is a self-supervised vision backbone whose features are so structured that objects pop out with **no labels**. We'll (1) run **PCA on its patch tokens** to visualise that, then (2) train a **linear probe** on its CLS feature for classification — the standard way to reuse a frozen foundation backbone.

> **Runtime → T4 GPU** recommended. Meant to run on Colab (downloads DINOv2 weights + data); not pre-executed here.

In [ ]:
!pip -q install transformers
import torch, requests, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
device = "cuda" if torch.cuda.is_available() else "cpu"
proc = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dino = AutoModel.from_pretrained("facebook/dinov2-small").to(device).eval()
print("DINOv2 ready on", device)

## 1 · Patch-feature PCA — objects emerge without labels

In [ ]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"   # two cats
img = Image.open(requests.get(url, stream=True).raw).convert("RGB")
inp = proc(images=img, return_tensors="pt").to(device)
with torch.no_grad():
    tokens = dino(**inp).last_hidden_state[0, 1:]               # drop CLS -> (num_patches, dim)
g = int(tokens.shape[0] ** 0.5)
U, S, V = torch.pca_lowrank(tokens, q=3)
pca = (tokens @ V[:, :3]); pca = (pca - pca.min(0).values) / (pca.max(0).values - pca.min(0).values + 1e-6)
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img); ax[0].set_title("input"); ax[0].axis("off")
ax[1].imshow(pca.reshape(g, g, 3).cpu()); ax[1].set_title("DINOv2 patch PCA (no labels)"); ax[1].axis("off")
plt.show()

## 2 · Linear probe on CLS features (CIFAR-10)

In [ ]:
from torchvision import datasets
import torch.nn as nn, random
test = datasets.CIFAR10(".", train=False, download=True); random.seed(0)
idxs = random.sample(range(len(test)), 800); feats, labels = [], []
with torch.no_grad():
    for i in idxs:
        im, lab = test[i]
        x = proc(images=im, return_tensors="pt").to(device)
        feats.append(dino(**x).pooler_output.cpu()); labels.append(lab)
X = torch.cat(feats); y = torch.tensor(labels)
clf = nn.Linear(X.shape[1], 10); opt = torch.optim.Adam(clf.parameters(), 1e-3)
for e in range(400):
    opt.zero_grad(); nn.functional.cross_entropy(clf(X[:600]), y[:600]).backward(); opt.step()
probe_acc = (clf(X[600:]).argmax(-1) == y[600:]).float().mean().item()
print("linear-probe accuracy:", probe_acc)

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/C_dinov2_features_probe/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/C_dinov2_features_probe"; os.makedirs(run, exist_ok=True)
torch.save(clf.state_dict(), f"{run}/probe.pt")
json.dump({"linear_probe": float(probe_acc)}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
Dense features feed **B** geometry and **D** mapping.

### Where to go next
- Run the PCA on **egocentric frames** (EPIC-Kitchens / Ego4D) — hands and manipulated objects separate cleanly.
- Use patch features for open-vocabulary segmentation, or as inputs to a hand/object detector (lessons C5–C6).
- Compare this probe with the **CLIP** probe lab — different pretraining, different features.